In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time

#### Reading data

In [0]:
df = spark.read.format("parquet")\
                .load("abfss://bronze@databrickspipeline07.dfs.core.windows.net/customers")
df.display()

#### Drop the column

In [0]:
df = df.drop("_rescued_data")
df.display()

#### Splitting domain from the email

In [0]:
df = df.withColumn("domain", split(col("email"), "@")[1])
df.display()

#### Finding the number of customers in each domains

In [0]:
df.groupBy("domain").agg(count("customer_id").alias("total_customers")).sort("total_customers", ascending=False).display()

#### Filtering gmail, yahoo, hotmail customers

In [0]:
df_gmail = df.filter(col("domain")=="gmail.com")
df_gmail.display()
time.sleep(2)

df_yahoo = df.filter(col("domain")=="yahoo.com")
df_yahoo.display()
time.sleep(5)

df_hotmail = df.filter(col("domain")=="hotmail.com")
df_hotmail.display()
time.sleep(5)

In [0]:
df = df.withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name")))
df = df.drop("first_name", "last_name")
df.display()

#### Writing data

In [0]:
df.write.mode("overwrite")\
        .format("delta")\
        .save("abfss://silver@databrickspipeline07.dfs.core.windows.net/customers")

#### Creating tables

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_pipeline_cata.silver.customers
USING DELTA
LOCATION 'abfss://silver@databrickspipeline07.dfs.core.windows.net/customers'

In [0]:
%sql
SELECT * FROM databricks_pipeline_cata.silver.customers